We will build a pure "research assistant" environment, with three tools:
    
    1. search(query) => search result text
    2. calculate(exp) => execute math calculation
    3. verify(fact) => verify fact (True/False)

In [2]:
from dataclasses import dataclass

@dataclass
class ToolResult:
    name: str
    input: str
    output: str
    success: bool

In [22]:
import re
class MiniToolEnv:
    KB = {
        "earth_radius": "6371",
        "pi": "3.14159265",
        "speed_of_light": "299792458",
        "gravity": "9.8",
        "moon_distance": "384400",
        "population_china": "1400000000",
        "python_release": "1991",
        "gpt_release": "2020",
        "transformer_paper": "2017",
    }

    def search(self,query:str)->ToolResult:
        query_lower = query.lower()
        for k,v in self.KB.items():
            if k in query_lower or any(w in k for w in query_lower.split("_")):
                return ToolResult(name="search",input=query,output=f"Found {k} = {v}", success=True)
        return ToolResult(name="search",input=query,output=f"No information found for {query}", success=False)
    def calculate(self,exp:str)->ToolResult:
        try:
            result = eval(exp)
            return ToolResult(name="calculate",input=exp,output=f"{exp} = {result}", success=True)
        except Exception as e:
            return ToolResult(name="calcualte",input=exp,output=f"Calculation error {e}", success=False)
    
    def verify(self,fact:str)->ToolResult:
        for k,v in self.KB.items():
            if k.replace("_"," ").strip() in fact.lower() and v in fact:
                return ToolResult(name="verify",input=fact,output=f"Correct", success=True)
        return ToolResult(name="search",input=fact,output=f"Wrong", success=False)



In [16]:
env = MiniToolEnv()
print(env.search("pi"))
print(env.calculate("2*3/2"))
print(env.verify("the value of pi is 3.14159265"))
print(env.verify("the value of pi is 3.14"))

ToolResult(name='search', input='pi', output='Found pi = 3.14159265', success=True)
ToolResult(name='calculate', input='2*3/2', output='2*3/2 = 3.0', success=True)
ToolResult(name='verify', input='the value of pi is 3.14159265', output='Correct', success=True)
ToolResult(name='search', input='the value of pi is 3.14', output='Wrong', success=False)


In [17]:
# Multi Turn Agent Loop
@dataclass
class Turn:
    action:str
    input: str
    observation:str
    success: bool

In [18]:
from typing import List
@dataclass
class Episode:
    task: str
    ground_truth: str
    turns: List[Turn]

In [23]:
def run_agent_loop(env:MiniToolEnv,task:str,action_plan:List[dict],ground_truth:str):
    turns = []
    for step in action_plan:
        tool = step["tool"]
        input = step["input"]

        if tool == "search":
            result = env.search(input)
        elif tool == "calculate":
            result = env.calculate(input)
        elif tool == "verify":
            result = env.verify(input)
        elif tool == "answer":
            correct = input.strip() == ground_truth.strip()
            turn = Turn("answer",input=input,observation="Correct!" if correct else "Wrong",success=correct)
            turns.append(turn)
            return Episode(task=task,ground_truth=ground_truth,turns=turns)
        else:
            result = ToolResult(tool,input,f"Unkown tool:{tool}",False)
        turn = Turn(tool,input,result.output,result.success)
        turns.append(turn)
    return Episode(task,ground_truth,turns)

In [25]:
from pprint import pprint 
task = "What is Earth's equatorial circumference in kilometers?"
ground_truth = "40030"

agent_good_plan = [
    {"tool":"search","input":"earth_radius"},
    {"tool":"calculate","input":"2 * 3.14159 * 6371"},
    {"tool":"verify","input":"earth radius is 6371"},
    {"tool":"answer","input":"40030"}
]
agent_bad_plan = [
    {"tool": "search", "input": "earth_radius"},      # Turn 1: correct search.
    {"tool": "calculate", "input": "2 * 3 * 6371"},   # Turn 2: pi is wrong.
    {"tool": "verify", "input": "earth radius is 6371"},   # Turn 3: correct verification.
    {"tool": "answer", "input": "38226"},              # Turn 4: wrong answer.
]

env = MiniToolEnv()
good_episode = run_agent_loop(env,task,agent_good_plan,ground_truth)
bad_episode = run_agent_loop(env,task,agent_bad_plan,ground_truth)
pprint(good_episode)
pprint(bad_episode)

Episode(task="What is Earth's equatorial circumference in kilometers?",
        ground_truth='40030',
        turns=[Turn(action='search',
                    input='earth_radius',
                    observation='Found earth_radius = 6371',
                    success=True),
               Turn(action='calculate',
                    input='2 * 3.14159 * 6371',
                    observation='2 * 3.14159 * 6371 = 40030.13978',
                    success=True),
               Turn(action='verify',
                    input='earth radius is 6371',
                    observation='Correct',
                    success=True),
               Turn(action='answer',
                    input='40030',
                    observation='Correct!',
                    success=True)])
Episode(task="What is Earth's equatorial circumference in kilometers?",
        ground_truth='40030',
        turns=[Turn(action='search',
                    input='earth_radius',
                    observation='F

In [40]:
def orm_credit(episode:Episode,gamma:float=0.95) -> List[float]:
    T = len(episode.turns)
    rewards = [0.0]*(T-1) + [1.0 if episode.turns[-1].success else -1.0]
    G = 0.0
    for i in reversed(range(T)):
        G = rewards[i] + gamma * G
        rewards[i] = G
    return rewards

In [41]:
orm_credit(bad_episode)

[-0.8573749999999999, -0.9025, -0.95, -1.0]

In [37]:
def prm_credit(episode:Episode,gamma:float=0.95)->List[float]:
    rewards = []
    for turn in episode.turns:
        action = turn.action
        success = turn.success
        if action == "answer":
            rewards.append(1.0 if success else -0.5)
        else:
            rewards.append(0.3 if success else -0.3)
    print(rewards)
    T = len(episode.turns)
    G = 0
    for i in reversed(range(T)):
        G = rewards[i] + gamma * G
        print(G)
        rewards[i] = G
    return rewards

In [38]:
prm_credit(bad_episode)

[0.3, 0.3, 0.3, -0.5]
-0.5
-0.175
0.13375
0.4270625


[0.4270625, 0.13375, -0.175, -0.5]